# Colab training run

This notebook runs the repository pipeline on the NASA TropicalCyclone imagery: install, download, manifest, classical features, baselines, then the CNN trainer.

It does not ship scores. Cells below have no saved outputs. A number printed after you run a cell is a validation score for that run, with `quote_as_result` left at no. Do not paste it into `docs/PROJECT_DOCUMENTATION.md`.

This is not a forecast, a warning, or an operational-readiness claim. Best-track agreement is not ground truth. ADT, SATCON, and Dvorak are not joined. The project class names are this project's own grouping, not official IMD classes. IMD uses 3-minute winds. NASA labels are HURDAT2 1-minute winds.

Use a CPU runtime. Do not enable a GPU runtime and do not use paid compute. The training code keeps tensors on CPU when `CUDA_VISIBLE_DEVICES` is empty.

The competition train and test sides share storms. `src.nasa_tc` keeps that side in `source_split` only. `src.baselines` then assigns a storm-wise split with `src.splits.assign_storms`. Do not train on `source_split` and score the other side.


In [ ]:
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ.pop("CLOUDIE_ALLOW_GPU", None)
os.environ.pop("CLOUDIE_ALLOW_PRETRAINED", None)
ROOT = Path.cwd()
assert (ROOT / "src" / "nasa_tc.py").is_file(), "Run this notebook from the repository root."
print("CPU only. CUDA_VISIBLE_DEVICES is empty. Not an operational system.")


## Install

TorchGeo's downloader needs the `azcopy` binary. The install cell does not download the imagery.


In [ ]:
%%bash
set -euo pipefail
pip install -q -r requirements.txt -r requirements-train.txt
pip install -q torchgeo
if ! command -v azcopy >/dev/null 2>&1; then
  curl -fsSL -o /tmp/azcopy.tgz https://aka.ms/downloadazcopy-v10-linux
  rm -rf /tmp/azcopy_extract
  mkdir -p /tmp/azcopy_extract
  tar -xzf /tmp/azcopy.tgz -C /tmp/azcopy_extract
  install -m 755 "$(find /tmp/azcopy_extract -type f -name azcopy | head -1)" /usr/local/bin/azcopy
fi
python -c "import os; assert os.environ.get('CUDA_VISIBLE_DEVICES', '') == ''"


## Download

`TropicalCyclone(..., download=True)` syncs the public Source Cooperative copy with azcopy. Images stay out of git. This step is large. It is the real NASA set, not the smoke images.


In [ ]:
import yaml
from pathlib import Path
from torchgeo.datasets import TropicalCyclone

config = yaml.safe_load(Path("config/config.yaml").read_text(encoding="utf-8"))
nasa_root = str(config["paths"]["nasa_root"])
for split in ("train", "test"):
    TropicalCyclone(root=nasa_root, split=split, download=True)
print("Download finished into", nasa_root, ". No scores yet.")


## Manifest

`python -m src.nasa_tc` builds the manifest. The file has no `split` column. Colour-enhanced frames are excluded. Chips stay at native size.


In [ ]:
%%bash
set -euo pipefail
python - <<'PY'
import yaml
from pathlib import Path
config = yaml.safe_load(Path("config/config.yaml").read_text(encoding="utf-8"))
root = Path(config["paths"]["nasa_root"])
Path("reports").mkdir(exist_ok=True)
print(root)
PY
NASA_ROOT="$(python -c 'import yaml; from pathlib import Path; print(yaml.safe_load(Path("config/config.yaml").read_text())["paths"]["nasa_root"])')"
python -m src.nasa_tc \
  --root "$NASA_ROOT" \
  --out "$NASA_ROOT/manifest.csv" \
  --reader csv \
  --summary reports/nasa_tc_summary.json \
  --sample-png reports/nasa_tc_sample.png
python - <<'PY'
import pandas as pd
import yaml
from pathlib import Path
root = Path(yaml.safe_load(Path("config/config.yaml").read_text())["paths"]["nasa_root"])
frame = pd.read_csv(root / "manifest.csv")
assert "split" not in frame.columns
assert "source_split" in frame.columns
assert not frame["is_synthetic"].any()
print("rows", len(frame), "storms", frame["storm_id"].nunique())
print("source_split counts")
print(frame.groupby("source_split")["storm_id"].nunique())
PY


## Features and baselines

Classical features come from `src.features`. The comparison is `src.baselines`: mean, persistence, ridge, ordinary least squares (`lr`), k-nearest neighbors, random forest, and XGBoost, plus RBF SVM (`svr` / `svc`), for wind in knots and for the five project classes.

Learned models are sklearn pipelines. SMOTE or class weights are applied inside the training fold only. Inner comparison uses `StratifiedGroupKFold` on `storm_id`. The test storms are not fit. Persistence is the previous same-storm wind and is not a feature.

The command writes `results/nasa_tc_baselines/`. Read `RUN.md` for the seed and the row and storm counts. `quote_as_result` stays no.


In [ ]:
%%bash
set -euo pipefail
export CUDA_VISIBLE_DEVICES=""
unset CLOUDIE_ALLOW_GPU
NASA_ROOT="$(python -c 'import yaml; from pathlib import Path; print(yaml.safe_load(Path("config/config.yaml").read_text())["paths"]["nasa_root"])')"
python -m src.baselines \
  --config config/config.yaml \
  --manifest "$NASA_ROOT/manifest.csv" \
  --out results/nasa_tc_baselines \
  --workers 4 \
  --feature-cache "$NASA_ROOT/feature_table.csv" \
  --manifest-command "python -m src.nasa_tc --root $NASA_ROOT --out $NASA_ROOT/manifest.csv --reader csv"


## CNN

Training is `python -m src.train` with `config/config.yaml`. NASA and TCIR are separate runs. The model is `simplecnn` or `resnet18`. This notebook does not define its own loop.

Every cell passes `--device cpu`. `CUDA_VISIBLE_DEVICES` is empty and `CLOUDIE_ALLOW_GPU` is unset, so a GPU is not used. Checkpoints and `metrics.json` under `artifacts/runs/` are gitignored. Nothing here is a filled-in result, and a finished run is not an operational product.

TCIR uses `paths.tcir_files` from the YAML. The entrypoint does not download those HDF5 files.


In [ ]:
%%bash
set -euo pipefail
export CUDA_VISIBLE_DEVICES=""
unset CLOUDIE_ALLOW_GPU
unset CLOUDIE_ALLOW_PRETRAINED
python -m src.train --config config/config.yaml --dataset nasa --model simplecnn --device cpu


In [ ]:
%%bash
set -euo pipefail
export CUDA_VISIBLE_DEVICES=""
unset CLOUDIE_ALLOW_GPU
unset CLOUDIE_ALLOW_PRETRAINED
python -m src.train --config config/config.yaml --dataset nasa --model resnet18 --device cpu


In [ ]:
%%bash
set -euo pipefail
export CUDA_VISIBLE_DEVICES=""
unset CLOUDIE_ALLOW_GPU
unset CLOUDIE_ALLOW_PRETRAINED
python -m src.train --config config/config.yaml --dataset tcir --model simplecnn --device cpu


In [ ]:
%%bash
set -euo pipefail
export CUDA_VISIBLE_DEVICES=""
unset CLOUDIE_ALLOW_GPU
unset CLOUDIE_ALLOW_PRETRAINED
python -m src.train --config config/config.yaml --dataset tcir --model resnet18 --device cpu
